# Семинар 3. Парсинг HTML: requests, BeautifulSoup, Selenium

**Парсинг** — автоматический сбор данных с веб-страниц. Сегодня мы соберём новости с **Lenta.ru**: заголовок, дату, рубрику, автора и текст, — а в лабораторной работе применим их для **мониторинга СМИ**.

**План:** HTML → BeautifulSoup → requests → одна карточка → вся страница → несколько страниц → текст новости → файл → Selenium.

**Правила вежливого парсинга:** собираем только открытые данные, делаем паузы между запросами, не скачиваем лишнего, проверяем правила сайта ([lenta.ru/robots.txt](https://lenta.ru/robots.txt)).

---
# 1. Как устроена веб-страница

Веб-страница — это текст на языке **HTML**. Он состоит из **тегов** — слов в угловых скобках, которые вкладываются друг в друга, как матрёшки:

```html
<a class="card-full-news" href="/news/2026/11/15/novost/">
    <h3 class="card-full-news__title">Заголовок новости</h3>
</a>
```

- `a`, `h3` — **имена тегов** (`a` — ссылка, `h3` — заголовок);
- `href` — **атрибут** с адресом ссылки;
- `class` — **класс**: название, которое разработчики дали элементу. По классам мы будем находить нужные данные.

**Как увидеть HTML нужного элемента:** откройте [lenta.ru/parts/news](https://lenta.ru/parts/news/), наведите курсор на заголовок новости → правая кнопка мыши → **«Просмотреть код»**. Подсвеченный тег — то, что нам нужно: смотрим его имя и класс.

---
# 2. BeautifulSoup: ищем нужное в HTML

**BeautifulSoup** разбирает HTML и находит в нём теги по имени и классу. Потренируемся на маленьком примере ленты новостей:

In [1]:
from bs4 import BeautifulSoup

html = """
<ul>
    <li><a class="card" href="/news/2026/11/15/brand/">
        <h3 class="title">Бренд запустил рекламную кампанию, созданную нейросетью</h3>
        <span class="rubric">Экономика</span>
    </a></li>
    <li><a class="card" href="/news/2026/11/15/smm/">
        <h3 class="title">Соцсети назвали главным каналом продвижения</h3>
        <span class="rubric">Интернет и СМИ</span>
    </a></li>
</ul>
"""

soup = BeautifulSoup(html, "html.parser")     # превращаем строку в «суп», в котором удобно искать

Три главных инструмента:

| Что | Как пишется |
|---|---|
| найти **первый** тег | `soup.find("тег", class_="класс")` — `class_` с подчёркиванием, слово `class` в Python занято |
| найти **все** теги (список) | `soup.find_all("тег", class_="класс")` |
| достать текст / атрибут | `.get_text()` / `тег["href"]` — как значение из словаря |

На настоящей странице 20 карточек. Поэтому сначала находим **все карточки**, а потом ищем данные **внутри каждой** — так заголовок одной новости не перепутается с рубрикой другой:

In [2]:
for card in soup.find_all("a", class_="card"):
    title = card.find("h3", class_="title").get_text()      # ищем внутри карточки, а не во всём супе
    rubric = card.find("span", class_="rubric").get_text()
    link = card["href"]                                      # атрибут — в квадратных скобках
    print(rubric, "|", title, "|", link)

Экономика | Бренд запустил рекламную кампанию, созданную нейросетью | /news/2026/11/15/brand/
Интернет и СМИ | Соцсети назвали главным каналом продвижения | /news/2026/11/15/smm/


---
# 3. requests: скачиваем страницу

**requests** получает с сайта HTML-код страницы — тот же текст, что и браузер. Заголовок `User-Agent` «представляет» нашу программу браузером: многие сайты не пускают программы без него.

Код ответа `200` — всё хорошо; `403` — доступ запрещён; `404` — страницы нет; `429` — слишком много запросов.

In [3]:
import requests

url = "https://lenta.ru/parts/news/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

response = requests.get(url, headers=headers)
print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")    # HTML страницы лежит в response.text
print(soup.title.get_text())                          # заголовок вкладки браузера

200
Новости: Lenta.ru


---
# 4. Разбираем одну карточку новости

Через «Просмотреть код» видно, что каждая новость в ленте устроена так:

```html
<a class="card-full-news _parts-news" href="/news/2026/11/15/coffee/">
    <h3 class="card-full-news__title">Отказ от кофе назвали причиной мигрени</h3>
    <time class="card-full-news__info-item card-full-news__date">14:05</time>
    <span class="card-full-news__info-item card-full-news__rubric">Забота о себе</span>
</a>
```

У тега может быть несколько классов через пробел — искать можно по любому из них.

⚠️ **Не все карточки в ленте — новости самой Lenta.ru.** Среди них бывают материалы партнёрского сайта «Мослента»: у них в `href` записан полный адрес чужого сайта (`https://moslenta.ru/...`), нет рубрики, а внутри другая разметка. У новостей Ленты адрес короткий и начинается с `/news/`. Поэтому сначала отберём только их — проверкой `startswith` («начинается с»):

In [4]:
cards = []
for card in soup.find_all("a", class_="card-full-news"):
    if card["href"].startswith("/news/"):          # только новости самой Lenta.ru
        cards.append(card)

print("Всего карточек:", len(soup.find_all("a", class_="card-full-news")), "| новостей Ленты:", len(cards))

Всего карточек: 20 | новостей Ленты: 18


Теперь разберём первую из них:

In [5]:
card = cards[0]                                                     # первая новость Ленты

title = card.find("h3", class_="card-full-news__title").get_text(strip=True)   # strip=True убирает лишние пробелы
news_time = card.find("time", class_="card-full-news__date").get_text(strip=True)
link = "https://lenta.ru" + card["href"]                            # в href нет адреса сайта — дописываем

# рубрика указана не у всех новостей: если тега нет, find вернёт None («ничего»)
rubric_tag = card.find("span", class_="card-full-news__rubric")
if rubric_tag is None:
    rubric = "без рубрики"
else:
    rubric = rubric_tag.get_text(strip=True)

# в карточке только время, а дата спрятана в ссылке: .../news/2026/11/15/...
parts = link.split("/")                                             # режем ссылку по «/», как отзывы в семинаре 2
date = parts[4] + "-" + parts[5] + "-" + parts[6]                   # год-месяц-день

print(title)
print(rubric, date, news_time)
print(link)

БПЛА атаковали промзону в Днепропетровской области
Бывший СССР 2026-11-15 23:41
https://lenta.ru/news/2026/11/15/bpla-atakovali-promzonu-v-dnepropetrovskoy-oblasti/


Два приёма на заметку:
- если данных не видно на странице, поищите их в адресе ссылки или в атрибутах тегов;
- если тега может не быть, проверяем `if тег is None`. Без проверки `get_text()` от «ничего» даст ошибку `'NoneType' object has no attribute 'get_text'`.

---
# 5. Собираем все новости со страницы

Соберём код из раздела 4 в **функцию**: она получает карточку и возвращает **словарь** с данными новости.

In [6]:
def parse_card(card):
    link = "https://lenta.ru" + card["href"]
    parts = link.split("/")

    rubric_tag = card.find("span", class_="card-full-news__rubric")
    if rubric_tag is None:
        rubric = "без рубрики"
    else:
        rubric = rubric_tag.get_text(strip=True)

    return {
        "title": card.find("h3", class_="card-full-news__title").get_text(strip=True),
        "link": link,
        "date": parts[4] + "-" + parts[5] + "-" + parts[6],
        "time": card.find("time", class_="card-full-news__date").get_text(strip=True),
        "rubric": rubric,
    }

Применим функцию ко всем карточкам. Отсутствие рубрики мы уже учли, но на странице может попасться и другая нестандартная карточка, на которой функция упадёт с ошибкой. Чтобы весь цикл не остановился, используем **`try` / `except`**: «попробуй, а если не вышло — пропусти».

In [7]:
news = []

for card in cards:                          # карточки Ленты, отобранные в разделе 4
    try:
        news.append(parse_card(card))
    except Exception:
        pass                                # нестандартную карточку пропускаем

print("Собрано новостей:", len(news))
for n in news[:5]:
    print(n["time"], "|", n["rubric"], "|", n["title"])

Собрано новостей: 18
23:41 | Бывший СССР | БПЛА атаковали промзону в Днепропетровской области
23:38 | Россия | В городском квартале в Курской области произошел взрыв по вине ВСУ
23:26 | Бывший СССР | Главе Приднестровья запретили выезд из Молдавии
23:23 | Мир | Американский министр отказался извиняться за транспортный коллапс
23:20 | Россия | Кремль отреагировал на недопуск молодежной сборной на чемпионат мира по хоккею


---
# 6. Проходим по нескольким страницам

Кнопка «Показать ещё» внизу ленты ведёт на адреса `/parts/news/2/`, `/parts/news/3/`… Меняется только номер, поэтому соберём адреса в цикле.

**Обязательно делаем паузу** `time.sleep()` между запросами, иначе сайт может нас заблокировать (код `429` или `403`).

In [8]:
import time

news = []

for page in range(1, 4):                                  # страницы 1, 2, 3
    if page == 1:
        url = "https://lenta.ru/parts/news/"
    else:
        url = f"https://lenta.ru/parts/news/{page}/"

    response = requests.get(url, headers=headers)
    print(url, "| код ответа:", response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")
    for card in soup.find_all("a", class_="card-full-news"):
        if card["href"].startswith("/news/"):             # только новости самой Lenta.ru
            try:
                news.append(parse_card(card))
            except Exception:
                pass

    time.sleep(2)                                         # пауза 2 секунды

print("Всего собрано новостей:", len(news))

https://lenta.ru/parts/news/ | код ответа: 200
https://lenta.ru/parts/news/2/ | код ответа: 200
https://lenta.ru/parts/news/3/ | код ответа: 200
Всего собрано новостей: 58


---
# 7. Заходим внутрь новости: полный текст

Для анализа текстов нужны не только заголовки, но и **полный текст**. На странице новости он разбит на абзацы:

```html
<div class="topic-body__content">
    <p class="topic-body__content-text">Первый абзац...</p>
    <p class="topic-body__content-text">Второй абзац...</p>
</div>
<span class="topic-authors__name">Имя автора</span>
```

Напишем функцию, которая открывает новость по ссылке и возвращает **текст и автора**:

In [9]:
def parse_article(link):
    response = requests.get(link, headers=headers, timeout=10)    # timeout: не ждём ответа дольше 10 секунд
    page = BeautifulSoup(response.text, "html.parser")

    # текст: находим все абзацы и склеиваем их через пробел
    body = page.find("div", class_="topic-body__content")
    texts = []
    for p in body.find_all("p", class_="topic-body__content-text"):
        texts.append(p.get_text().strip())
    text = " ".join(texts)

    # автор: указан не у всех новостей; если тега нет, find вернёт None
    author_tag = page.find("span", class_="topic-authors__name")
    if author_tag is None:
        author = ""
    else:
        author = author_tag.get_text(strip=True)

    return text, author

text, author = parse_article(news[0]["link"])      # функция возвращает два значения сразу
print("Автор:", author)
print(text[:300])

Автор: Марина Совина
Премьер-министр Венгрии Петер Мадьяр назвал своего предшественника Виктора Орбана «падшим боссом мафии» из-за найденного в лесу тайника с деньгами бывшего министра национального развития Венгрии Миклоша Шестака. Об этом он написал в социальной сети Х. «Виктор Орбан, босс мафии, в настоящее время скр


Что здесь нового:
- `" ".join(список)` — склеивает строки из списка через пробел. Это противоположность `split()`.
- `if author_tag is None` — та же проверка «тег не нашёлся», что с рубрикой в разделе 4.
- Почему `p.get_text().strip()`, а не `get_text(strip=True)`? Внутри абзацев есть ссылки: `Жителей <a>Ульяновска</a> возмутил`. `strip=True` обрезает пробелы у **каждого кусочка** и склеивает слова: «ЖителейУльяновскавозмутил».

Теперь допишем текст и автора к первым 10 новостям (срез `news[:10]`). Это займёт около 15 секунд.

In [10]:
for n in news[:10]:                                # для скорости — только первые 10 новостей
    try:
        text, author = parse_article(n["link"])
    except Exception:                              # у фотогалерей и спецпроектов нет обычного текста
        text, author = "", ""
    n["text"] = text                               # добавляем в словарь новости новые ключи
    n["author"] = author
    print(n["title"], "| автор:", author)
    time.sleep(0.5)

Мадьяр назвал Орбана «падшим боссом мафии» из-за найденных в лесу миллиардов | автор: Марина Совина
БПЛА атаковали промзону в Днепропетровской области | автор: Евгений Силаев
В городском квартале в Курской области произошел взрыв по вине ВСУ | автор: Александра Синицына
Главе Приднестровья запретили выезд из Молдавии | автор: Евгений Силаев
Американский министр отказался извиняться за транспортный коллапс | автор: Юлия Мискевич
Кремль отреагировал на недопуск молодежной сборной на чемпионат мира по хоккею | автор: Марина Совина
Зеленский сделал признание после ударов России | автор: Марина Совина
Эстония задумала закрыть все погранпереходы с Россией с 1 декабря. Что послужило причиной и как отреагировали в Москве? | автор: Александра Синицына
Психолог назвала способы помочь близкому при панической атаке | автор: Марина Совина
Олимпийский чемпион объяснил недопуск Большунова к международным турнирам | автор: Анастасия Борисова


---
# 8. Сохраняем результат в файл

Данные на сайтах живые: новости правят и удаляют. Поэтому собранное сразу сохраняем в файл вместе с **датой сбора**. Модуль `csv` сам правильно обработает точки с запятой внутри текста.

In [11]:
import csv
from datetime import date

with open("lenta_news.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["date", "time", "rubric", "author", "title", "text", "link", "collected"])
    for n in news[:10]:                            # первые 10 новостей — у них есть текст
        writer.writerow([n["date"], n["time"], n["rubric"], n["author"],
                         n["title"], n["text"], n["link"], str(date.today())])

print("Файл сохранён")

Файл сохранён


Файл `lenta_news.csv` появится в панели **Файлы** слева: правая кнопка мыши → **Скачать**.

---
# 9. Selenium: когда requests не справляется

*Демонстрация преподавателя.*

`requests` получает страницу такой, какой её **прислал сервер**. Но часть содержимого многие сайты дорисовывают уже в браузере с помощью **JavaScript** — при прокрутке или нажатии кнопок. `requests` JavaScript не выполняет и таких данных не видит.

**Selenium** управляет настоящим браузером: открывает страницу, нажимает кнопки, прокручивает — как человек.

**Как понять, что нужно:** откройте страницу, нажмите **Ctrl+U** (исходный код) и найдите нужный текст через **Ctrl+F**. Нашёлся — хватит `requests`. Нет — нужен Selenium.

В Colab нет браузера, поэтому сначала устанавливаем Chrome и Selenium (около минуты). `!` в начале строки — это команда для облачного компьютера, а не код Python.

In [12]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb > /dev/null
!pip install -q selenium

Запускаем браузер **без окна** (`headless`), нажимаем «Показать ещё» три раза — как человек, который листает ленту, — и сравниваем часы в шапке сайта.

Поиск в Selenium похож на BeautifulSoup: `driver.find_elements(By.CSS_SELECTOR, "h3.card-full-news__title")` — найти все теги `h3` с этим классом (точка означает «класс»), а текст элемента — `.text`.

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")             # браузер работает без окна
options.add_argument("--no-sandbox")               # служебные настройки,
options.add_argument("--disable-dev-shm-usage")    # без которых Chrome не запускается в Colab

driver = webdriver.Chrome(options=options)         # запускаем браузер
driver.get("https://lenta.ru/parts/news/")         # открываем страницу
time.sleep(3)                                      # ждём, пока загрузится

# 1. Часы в шапке сайта: в исходном HTML тег пустой, время вписывает JavaScript.
#    Смотрим сразу после загрузки, пока шапка на экране.
soup = BeautifulSoup(requests.get("https://lenta.ru/parts/news/", headers=headers).text, "html.parser")
print("requests — часы в шапке: [" + soup.find("time", class_="header__time").get_text() + "]")

clock = driver.find_element(By.CSS_SELECTOR, "time.header__time")
print("Selenium — часы в шапке: [" + clock.get_attribute("textContent") + "]")   # textContent — весь текст тега
print()

# 2. Нажимаем «Показать ещё» три раза, как человек, который листает ленту
titles = driver.find_elements(By.CSS_SELECTOR, "h3.card-full-news__title")
print("Новостей до нажатия:", len(titles))

for click in range(3):
    button = driver.find_element(By.CSS_SELECTOR, "a.loadmore")        # кнопка «Показать ещё»
    driver.execute_script("arguments[0].scrollIntoView();", button)    # прокручиваем к ней
    driver.execute_script("arguments[0].click();", button)             # нажимаем
    time.sleep(3)                                                      # ждём подгрузки
    titles = driver.find_elements(By.CSS_SELECTOR, "h3.card-full-news__title")
    print("После нажатия", click + 1, "— новостей:", len(titles))

driver.quit()                                      # закрываем браузер

requests — часы в шапке: []
Selenium — часы в шапке: [15 ноября, 23:54]

Новостей до нажатия: 20
После нажатия 1 — новостей: 40
После нажатия 2 — новостей: 60
После нажатия 3 — новостей: 80


Новостей становится больше, хотя **адрес страницы тот же**, а часы видит только Selenium: всё это дорисовал JavaScript.

**Вывод:** `requests` видит страницу такой, какой её прислал сервер, а Selenium — такой, какой её видит человек. Selenium медленнее и сложнее, поэтому сначала всегда пробуем `requests`.

---
# Итоги

| Что | Как пишется |
|---|---|
| Скачать страницу | `requests.get(url, headers=headers)` |
| Проверить ответ | `response.status_code` → `200` |
| Сделать «суп» | `BeautifulSoup(response.text, "html.parser")` |
| Найти первый / все теги | `soup.find(...)` / `soup.find_all(...)` |
| Текст / атрибут тега | `.get_text(strip=True)` / `тег["href"]` |
| Не падать на ошибке | `try:` … `except Exception:` |
| Пауза между запросами | `time.sleep(2)` |
| Сохранить в CSV | `csv.writer(f, delimiter=";")` |
| Страница с JavaScript | Selenium |

**Алгоритм:** «Просмотреть код» → найти теги и классы → скачать страницу → найти карточки (`find_all`) → внутри каждой данные (`find`) → функция + цикл по страницам с паузами → сохранить в файл.